<h1 style="color:#8B0000; font-family:Georgia,serif;">🩺 Chronic Disease ETL Pipeline</h1>
<p style="color:#2F4F4F; font-size:15px;">
This notebook cleans and consolidates Nova Scotia Health Authority (NSHA) chronic disease datasets 
into a single standardised output for downstream analysis and Power BI reporting.
</p>

---

<h2 style="color:#8B0000;">📦 Step 1 — Import Libraries</h2>
<p style="color:#2F4F4F;">Load <code>pandas</code> for all data wrangling operations.</p>

In [1]:
import pandas as pd

---

<h2 style="color:#8B0000;">🔧 Step 2 — Helper Functions</h2>

<h3 style="color:#B22222;">split_zone()</h3>
<p style="color:#2F4F4F;">
Parses a raw zone string like <code>"Zone 4 - Central"</code> into two separate fields: 
<strong>Health Zone</strong> (e.g. <code>Zone 4</code>) and <strong>Health Zone Name</strong> (e.g. <code>Central</code>).  
This normalises the zone column to match the <code>DimHealthZone</code> schema in the star schema data warehouse.
</p>

<h3 style="color:#B22222;">clean_chronic_file()</h3>
<p style="color:#2F4F4F;">
A generic cleaner that handles <strong>any</strong> NSHA chronic disease CSV. Parameters:
</p>

| Parameter | Description |
|---|---|
| `file_path` | Path to the source CSV file |
| `disease_name` | Label to tag rows with (e.g. `'Diabetes'`) |
| `prevalence_column` | Name of the prevalence count column in the source (varies by file) |
| `force_gender` | Overrides gender column when source doesn't have M/F breakdown (e.g. AMI) |

<p style="color:#2F4F4F;">The function returns a standardised DataFrame with this schema:</p>

| Column | Type | Notes |
|---|---|---|
| `Date` | datetime | Parsed from year field |
| `agegroup` | string | Age band e.g. `20 to 29` |
| `Gender` | string | `M`, `F`, or `All` |
| `Health Zone` | string | Zone code e.g. `Zone 4` |
| `Health Zone Name` | string | Zone label e.g. `Central` |
| `Chronic Disease` | string | Disease label |
| `prevalence` | int | Count of cases |
| `population` | int | Reference population |

---

<h2 style="color:#8B0000;">🗂️ Step 3 — Load & Clean Each Disease</h2>
<p style="color:#2F4F4F;">Each source CSV is passed through <code>clean_chronic_file()</code>. Six diseases are loaded:</p>

| Disease | Source File | Notes |
|---|---|---|
| **Diabetes** | `Diabetes Crude Prevalence.csv` | Standard M/F breakdown |
| **Hypertension** | `Hypertension Crude Prevalence.csv` | Column named `hypertension_count` |
| **Ischemic Heart Disease** | `Ischemic Heart Disease Crude Prevalence.csv` | Standard M/F breakdown |
| **COPD** | `Chronic Obstructive Pulmonary Disease (COPD) Crude Prevalence.csv` | Standard M/F breakdown |
| **AMI** | Reuses IHD file | Gender forced to `All` (no M/F split available) |
| **Asthma** | Reuses IHD file | Standard M/F breakdown |

---

<h2 style="color:#8B0000;">🔗 Step 4 — Combine, Validate & Export</h2>
<p style="color:#2F4F4F;">
All six disease DataFrames are stacked with <code>pd.concat()</code> into one unified dataset.  
Row counts are printed per disease for validation, then the result is saved to:
</p>
<p style="color:#8B0000; font-weight:bold; font-size:14px;">📁 <code>./clean/chronic_diseases.csv</code></p>

In [4]:

# ---------------------------------------------------
# Helper: Split Health Zone Code & Name
# ---------------------------------------------------
def split_zone(zone_str):
    """
    'Zone 4 - Central' -> ('Zone 4', 'Central')
    """
    zone_code, zone_name = zone_str.split('-', 1)
    return zone_code.strip(), zone_name.strip()


# ---------------------------------------------------
# Generic Cleaner
# ---------------------------------------------------
def clean_chronic_file(file_path, disease_name, prevalence_column, force_gender=None):
    df = pd.read_csv(file_path)

    # Normalize column names
    df.columns = df.columns.str.lower().str.strip()

    # Standardize column names
    df = df.rename(columns={
        'sex': 'gender',
        'age_group': 'agegroup',
        'year': 'date',
        prevalence_column: 'prevalence'
    })

    # Date
    df['date'] = pd.to_datetime(df['date'])

    # Gender handling
    if force_gender is not None:
        df['gender'] = force_gender
    else:
        df['gender'] = df['gender'].replace({
            'M': 'M',
            'F': 'F',
            'All': 'All'
        })

    # Split zone fields
    df[['Health Zone', 'Health Zone Name']] = (
        df['zone'].apply(lambda x: pd.Series(split_zone(x)))
    )

    # Add disease label
    df['Chronic Disease'] = disease_name

    # Final schema
    df = df[[
        'date',
        'agegroup',
        'gender',
        'Health Zone',
        'Health Zone Name',
        'Chronic Disease',
        'prevalence',
        'population'
    ]]

    # Rename columns
    df = df.rename(columns={
        'date': 'Date',
        'gender': 'Gender'
    })

    return df


# ---------------------------------------------------
# Existing chronic diseases
# ---------------------------------------------------
diabetes_df = clean_chronic_file(
    './data/Diabetes Crude Prevalence.csv',
    'Diabetes',
    'prevalence'
)

hypertension_df = clean_chronic_file(
    './data/Hypertension Crude Prevalence.csv',
    'Hypertension',
    'hypertension_count'
)

ihd_df = clean_chronic_file(
    './data/Ischemic Heart Disease Crude Prevalence.csv',
    'Ischemic Heart Disease',
    'prevalence'
)

copd_df = clean_chronic_file(
    './data/Chronic Obstructive Pulmonary Disease (COPD) Crude Prevalence.csv',
    'Chronic Obstructive Pulmonary Disease',
    'prevalence'
)
# ---------------------------------------------------
# NEW: Acute Myocardial Infarction (AMI → Gender = All)
# ---------------------------------------------------
ami_df = clean_chronic_file(
    './data/Ischemic Heart Disease Crude Prevalence.csv',
    'Acute Myocardial Infarction (AMI)',
    'prevalence',
    force_gender='All'
)

ami_df = ami_df[ami_df['Chronic Disease'] == 'Acute Myocardial Infarction (AMI)']


# ---------------------------------------------------
# NEW: Asthma (gender already exists)
# ---------------------------------------------------
asthma_df = clean_chronic_file(
    './data/Ischemic Heart Disease Crude Prevalence.csv',
    'Asthma',
    'prevalence'
)

asthma_df = asthma_df[asthma_df['Chronic Disease'] == 'Asthma']


# ---------------------------------------------------
# Append into ONE dataset
# ---------------------------------------------------
chronic_diseases = pd.concat(
    [
        diabetes_df,
        hypertension_df,
        ihd_df,
        ami_df,
        copd_df,
        asthma_df
    ],
    ignore_index=True
)


# ---------------------------------------------------
# Final validation
# ---------------------------------------------------
print(chronic_diseases['Chronic Disease'].value_counts())
display(chronic_diseases.head())


# ---------------------------------------------------
# Save output
# ---------------------------------------------------
chronic_diseases.to_csv(
    './clean/chronic_diseases.csv',
    index=False
)

Chronic Disease
Diabetes                                 1288
Hypertension                             1288
Ischemic Heart Disease                   1288
Acute Myocardial Infarction (AMI)        1288
Asthma                                   1288
Chronic Obstructive Pulmonary Disease     920
Name: count, dtype: int64


,Date,agegroup,Gender,Health Zone,Health Zone Name,Chronic Disease,prevalence,population
0,2000-01-01,20 to 29,F,Zone 4,Central,Diabetes,223,30198
1,2000-01-01,30 to 39,F,Zone 4,Central,Diabetes,455,34639
2,2000-01-01,40 to 49,F,Zone 4,Central,Diabetes,934,32980
3,2000-01-01,50 to 59,F,Zone 4,Central,Diabetes,1667,24173
4,2000-01-01,60 to 69,F,Zone 4,Central,Diabetes,1818,14990
